In [1]:
# run_raster_parallel.py
from __future__ import annotations

import os
import multiprocessing as mp
import concurrent.futures as cf
from pathlib import Path

from general_utils import find_ephys_sessions
from raster_worker import process_session, OUTDIR  # import the top-level worker

def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)
    _, _, sessions = find_ephys_sessions()
    sessions=['ecephys_795396_2025-09-20_13-11-19_sorted_2025-10-29_20-43-11']
    print("Sessions:", sessions)
    if not sessions:
        return

    # Use spawn to avoid HDF5/NWB fork-safety issues
    ctx = mp.get_context("spawn")
    max_workers = min(len(sessions), os.cpu_count() or 1)
    print(f"Running in parallel with {max_workers} workers (spawn)")

    with cf.ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as ex:
        futures = {ex.submit(process_session, s): s for s in sessions}
        for fut in cf.as_completed(futures):
            print(fut.result())

if __name__ == "__main__":
    # If something else already set the start method, this is fine.
    try:
        mp.set_start_method("spawn", force=False)
    except RuntimeError:
        pass
    main()


Sessions: ['ecephys_795396_2025-09-20_13-11-19_sorted_2025-10-29_20-43-11']
Running in parallel with 1 workers (spawn)

[ecephys_795396_2025-09-20_13-11-19_sorted_2025-10-29_20-43-11] models: ['QLearning_L1F0_epsi', 'WSLS', 'QLearning_L1F1_CK1_softmax', 'ForagingCompareThreshold', 'QLearning_L2F1_softmax', 'QLearning_L2F1_CKfull_softmax']
Found ephys NWB: /root/capsule/data/ecephys_795396_2025-09-20_13-11-19_sorted_2025-10-29_20-43-11/nwb/ecephys_795396_2025-09-20_13-11-19_experiment1_recording1.nwb
Successfully read ephys NWB from: /root/capsule/data/ecephys_795396_2025-09-20_13-11-19_sorted_2025-10-29_20-43-11/nwb/ecephys_795396_2025-09-20_13-11-19_experiment1_recording1.nwb
Found behavior NWB: /root/capsule/data/optogenetics_nwb/795396_2025-09-20_13-11-19.nwb
Successfully read behavior NWB from: /root/capsule/data/optogenetics_nwb/795396_2025-09-20_13-11-19.nwb
Successfully appended units table to behavior NWB.
Saved Unit 0 figure to /root/capsule/scratch/raster_plot/ecephys_795396_